# 🧠 Daily Challenge: Classification with Neural Networks in TensorFlow

**What You'll Learn:**
- Understand the different types of classification: Binary, Multi-class, and Multi-label
- Build Neural Networks for Classification using TensorFlow
- Improve model performance with layers, neurons, and activation functions
- Visualize decision boundaries to understand model predictions

---

## Step 1: Understanding Classification Types

### 🔵 Binary Classification
**Definition:** The model predicts one of **two possible classes** (0 or 1, Yes or No, True or False).

**Example:** Email Spam Detection — an email is either *spam* or *not spam*. The model outputs a single probability score; if > 0.5, it's spam.

---

### 🟢 Multi-class Classification
**Definition:** The model predicts **one class out of three or more possible classes** (mutually exclusive — only one label per sample).

**Example:** Handwritten Digit Recognition (MNIST) — a digit image is classified as exactly one of: 0, 1, 2, 3, 4, 5, 6, 7, 8, or 9.

---

### 🟡 Multi-label Classification
**Definition:** The model predicts **multiple labels simultaneously** for a single sample. Labels are NOT mutually exclusive — a sample can belong to more than one class at the same time.

**Example:** Movie Genre Tagging — a single movie can be tagged as *Action*, *Comedy*, AND *Romance* all at once.

---

| Type | Output Classes | Example |
|------|---------------|----------|
| Binary | 2 | Spam / Not Spam |
| Multi-class | 3+ (mutually exclusive) | Digit 0–9 |
| Multi-label | 3+ (non-exclusive) | Movie genres |

## Step 2: Set Up Environment & Dataset

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print("✅ All libraries imported successfully!")

In [ ]:
# Create the dataset
samples = 1000
X, y = make_circles(samples,
                    noise=0.03,
                    random_state=42)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('\nFirst 5 samples of X:\n', X[:5])
print('\nFirst 5 labels of y:', y[:5])
print(f'\nClass distribution — Class 0: {(y==0).sum()}, Class 1: {(y==1).sum()}')

In [ ]:
# Visualize the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', label='Class 0 (Outer)', alpha=0.7, edgecolors='k', linewidths=0.5)
plt.scatter(X[y==1, 0], X[y==1, 1], c='tomato', label='Class 1 (Inner)', alpha=0.7, edgecolors='k', linewidths=0.5)
plt.title('📊 Dataset Visualization: make_circles', fontsize=14, fontweight='bold')
plt.xlabel('Feature X1')
plt.ylabel('Feature X2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("💡 Observation: The two classes form concentric circles — a non-linear boundary is needed!")

## Step 3: Build a Basic Neural Network Model

In [ ]:
# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Build a basic model — just 1 dense layer
model_v1 = tf.keras.Sequential([
    tf.keras.layers.Dense(1, input_shape=(2,))
], name='Basic_Model_v1')

# Compile with SGD optimizer and Binary Crossentropy loss
model_v1.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.SGD(),
    metrics=['accuracy']
)

model_v1.summary()

In [ ]:
# Train the basic model
history_v1 = model_v1.fit(
    X, y,
    epochs=100,
    verbose=0  # silent training
)

# Evaluate
loss_v1, acc_v1 = model_v1.evaluate(X, y, verbose=0)
print(f"📌 Basic Model (v1) — Loss: {loss_v1:.4f} | Accuracy: {acc_v1*100:.2f}%")
print("⚠️  A single layer without activation struggles with non-linear data!")

## Step 4: Improve the Model — More Layers, Neurons & Adam Optimizer

In [ ]:
tf.random.set_seed(42)

# Improved model: more layers, more neurons, Adam optimizer
model_v2 = tf.keras.Sequential([
    tf.keras.layers.Dense(10, input_shape=(2,), activation='relu'),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
], name='Improved_Model_v2')

model_v2.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

model_v2.summary()

In [ ]:
# Train for more epochs
history_v2 = model_v2.fit(
    X, y,
    epochs=200,
    verbose=0
)

loss_v2, acc_v2 = model_v2.evaluate(X, y, verbose=0)
print(f"✅ Improved Model (v2) — Loss: {loss_v2:.4f} | Accuracy: {acc_v2*100:.2f}%")
print(f"📈 Accuracy improvement: +{(acc_v2 - acc_v1)*100:.2f}%")

In [ ]:
# Plot training history for v2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history_v2.history['loss'], color='tomato', linewidth=2)
axes[0].set_title('Training Loss (Model v2)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history_v2.history['accuracy'], color='steelblue', linewidth=2)
axes[1].set_title('Training Accuracy (Model v2)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Visualize the Decision Boundary

In [ ]:
def plot_decision_boundary(model, X, y, title='Decision Boundary', ax=None):
    """
    Plots the decision boundary of a trained binary classification model.
    
    Args:
        model: Trained TensorFlow/Keras model
        X: Feature array (n_samples, 2)
        y: Labels array (n_samples,)
        title: Plot title
        ax: Optional matplotlib axes object
    """
    # Define grid boundaries with some padding
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1

    # Create a mesh grid
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    # Predict on the grid
    X_grid = np.c_[xx.ravel(), yy.ravel()]
    y_pred = model.predict(X_grid, verbose=0).reshape(xx.shape)

    # Plot
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 6))

    ax.contourf(xx, yy, y_pred, cmap=plt.cm.RdBu, alpha=0.4, levels=50)
    ax.contour(xx, yy, y_pred, levels=[0.5], colors='black', linewidths=1.5)
    ax.scatter(X[y==0, 0], X[y==0, 1], c='steelblue', label='Class 0', alpha=0.8, edgecolors='k', linewidths=0.4)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='tomato', label='Class 1', alpha=0.8, edgecolors='k', linewidths=0.4)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature X1')
    ax.set_ylabel('Feature X2')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.2)

print("✅ plot_decision_boundary() defined!")

In [ ]:
# Compare decision boundaries: Basic vs Improved model
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_decision_boundary(model_v1, X, y, title='Basic Model v1 (1 layer, SGD)', ax=axes[0])
plot_decision_boundary(model_v2, X, y, title='Improved Model v2 (3 layers, ReLU, Adam)', ax=axes[1])

plt.suptitle('Decision Boundary Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("💡 v2's curved boundary shows it successfully learned the circular pattern!")

## Step 6: Incorporate Activation Functions — ReLU vs Sigmoid

In [ ]:
# Visualize activation functions
x_vals = np.linspace(-5, 5, 200)

relu   = np.maximum(0, x_vals)
sigmoid = 1 / (1 + np.exp(-x_vals))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(x_vals, relu, color='tomato', linewidth=2.5)
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].axvline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title('ReLU: f(x) = max(0, x)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)
axes[0].text(2, 0.5, 'Output range: [0, ∞)', fontsize=10, color='tomato')

axes[1].plot(x_vals, sigmoid, color='steelblue', linewidth=2.5)
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Sigmoid: f(x) = 1 / (1 + e⁻ˣ)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('x'); axes[1].set_ylabel('f(x)')
axes[1].grid(True, alpha=0.3)
axes[1].text(-4.5, 0.75, 'Output range: (0, 1)', fontsize=10, color='steelblue')

plt.suptitle('Activation Functions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
tf.random.set_seed(42)

# Model explicitly using ReLU in hidden layers + Sigmoid on output
model_v3 = tf.keras.Sequential([
    tf.keras.layers.Dense(16, input_shape=(2,), activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8,  activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')   # sigmoid → probability output
], name='Model_v3_ReLU_Sigmoid')

model_v3.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

model_v3.summary()

In [ ]:
history_v3 = model_v3.fit(X, y, epochs=300, verbose=0)

loss_v3, acc_v3 = model_v3.evaluate(X, y, verbose=0)
print(f"🚀 Model v3 (ReLU+Sigmoid, 4 layers) — Loss: {loss_v3:.4f} | Accuracy: {acc_v3*100:.2f}%")

# Summary comparison table
print("\n" + "="*55)
print(f"{'Model':<25} {'Loss':>8} {'Accuracy':>10}")
print("="*55)
print(f"{'v1 (Basic, SGD)':<25} {loss_v1:>8.4f} {acc_v1*100:>9.2f}%")
print(f"{'v2 (3L, ReLU, Adam)':<25} {loss_v2:>8.4f} {acc_v2*100:>9.2f}%")
print(f"{'v3 (4L, ReLU+Sig)':<25} {loss_v3:>8.4f} {acc_v3*100:>9.2f}%")
print("="*55)

## Step 7: Split Data into Training and Testing Sets (80/20)

In [ ]:
# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

# Visualize the split
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, X_s, y_s, name in zip(axes,
                               [X_train, X_test],
                               [y_train, y_test],
                               ['Training Set (80%)', 'Test Set (20%)']):
    ax.scatter(X_s[y_s==0, 0], X_s[y_s==0, 1], c='steelblue', label='Class 0', alpha=0.7, edgecolors='k', linewidths=0.4)
    ax.scatter(X_s[y_s==1, 0], X_s[y_s==1, 1], c='tomato', label='Class 1', alpha=0.7, edgecolors='k', linewidths=0.4)
    ax.set_title(f'{name} — {X_s.shape[0]} samples', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature X1'); ax.set_ylabel('Feature X2')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
tf.random.set_seed(42)

# Final model — train on training set only
model_final = tf.keras.Sequential([
    tf.keras.layers.Dense(16, input_shape=(2,), activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8,  activation='relu'),
    tf.keras.layers.Dense(1,  activation='sigmoid')
], name='Final_Model')

model_final.compile(
    loss='binary_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    metrics=['accuracy']
)

# Train on training data
history_final = model_final.fit(
    X_train, y_train,
    epochs=300,
    validation_data=(X_test, y_test),
    verbose=0
)

print("✅ Final model trained on 80% training data!")

## Step 8: Evaluate & Visualize Final Model Performance

In [ ]:
# Evaluate on train and test sets
train_loss, train_acc = model_final.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc   = model_final.evaluate(X_test, y_test, verbose=0)

print("="*45)
print(f"  Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc*100:.2f}%")
print(f"  Test  Loss: {test_loss:.4f}  | Test  Accuracy: {test_acc*100:.2f}%")
print("="*45)

In [ ]:
# Plot training & validation curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history_final.history['loss'],     label='Train Loss',  color='steelblue', linewidth=2)
axes[0].plot(history_final.history['val_loss'], label='Val Loss',    color='tomato',    linewidth=2, linestyle='--')
axes[0].set_title('Loss Curves — Final Model', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history_final.history['accuracy'],     label='Train Accuracy', color='steelblue', linewidth=2)
axes[1].plot(history_final.history['val_accuracy'], label='Val Accuracy',   color='tomato',    linewidth=2, linestyle='--')
axes[1].set_title('Accuracy Curves — Final Model', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Decision boundaries on Train vs Test
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

plot_decision_boundary(
    model_final, X_train, y_train,
    title=f'Final Model — Training Set\nAcc: {train_acc*100:.2f}%',
    ax=axes[0]
)
plot_decision_boundary(
    model_final, X_test, y_test,
    title=f'Final Model — Test Set\nAcc: {test_acc*100:.2f}%',
    ax=axes[1]
)

plt.suptitle('🔍 Decision Boundaries: Train vs Test', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Final model comparison across all versions
models    = ['v1\n(Basic, SGD)', 'v2\n(3L, Adam)', 'v3\n(4L, Full data)', 'Final\n(4L, Train/Test)']
accs      = [acc_v1, acc_v2, acc_v3, test_acc]
colors    = ['#b0c4de', '#87ceeb', '#4682b4', '#1e3a5f']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(models, [a*100 for a in accs], color=colors, edgecolor='black', linewidth=0.8, width=0.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('📊 Model Accuracy Progression', fontsize=14, fontweight='bold')
ax.axhline(y=90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Step 9: Key Takeaways & Summary

### 📝 What I Learned

**1. Classification Types Matter**  
Understanding whether your problem is binary, multi-class, or multi-label determines your output layer design, loss function, and final activation function.

**2. Simple Models Fail on Non-Linear Data**  
A single dense layer (model v1) achieved poor accuracy (~50%) because concentric circles require a non-linear decision boundary — something a single linear layer simply cannot produce.

**3. Depth + Activation Functions = Power**  
Adding hidden layers with **ReLU** activation functions dramatically improved performance. ReLU introduces non-linearity that lets the model learn curved decision boundaries. The **Sigmoid** output maps predictions to probabilities (0–1), which is ideal for binary classification.

**4. Optimizer Choice is Critical**  
Switching from **SGD** to **Adam** accelerated convergence and boosted accuracy significantly. Adam adapts the learning rate per parameter — a major advantage over plain gradient descent.

**5. Visualizing Decision Boundaries is Invaluable**  
Plotting `plot_decision_boundary()` revealed *why* models fail or succeed — not just *that* they do. This bridges the gap between a numeric accuracy score and actual model understanding.

**6. Train/Test Split Prevents Overconfidence**  
Training on 100% of data inflates reported accuracy. Using an 80/20 train-test split gives an honest view of how the model generalizes to unseen data.

---

### 🔑 Key Hyperparameters That Improved Performance

| Hyperparameter | v1 (Baseline) | Final Model |
|----------------|--------------|-------------|
| Layers | 1 | 4 |
| Neurons/layer | 1 | 16, 16, 8, 1 |
| Activation | None | ReLU + Sigmoid |
| Optimizer | SGD | Adam |
| Epochs | 100 | 300 |
| Data split | None | 80/20 train/test |

---

> 💡 **Bottom Line:** Neural networks for classification are powerful, but success depends on choosing the right architecture, activations, optimizer, and evaluating honestly with held-out test data. Always *visualize* — numbers alone don't tell the full story!